# quantum-cortex — the 8-seed run: pure recurrent ladder + control + 2 hybrids (RESUMABLE)

**128 units** = 8 paths (5 pure rungs L0–L4, the transformer control, the Hopfield hybrid, the delta hybrid) × 8 seeds × 2 tiers, each unit written to disk the moment it finishes. Every comparison is **paired on seed** at identical steps/tiers. **A quota cut loses at most one unit.**

Estimated ~10 h on T4 at 1500 steps (the delta hybrid's sequential scan dominates). Fits one 12 h session with `--time-budget-min 700`; if cut, resume.

## How to resume after a cut
1. This version's Output keeps `metrics/mqar/ladder8-ckpt/` (the finished units).
2. New version → **Add Input → Your Work → this notebook's previous version**. Kaggle mounts it under `/kaggle/input/notebooks/<user>/<notebook>/quantum-cortex/…`; the detection cell finds it recursively. Run that cell alone first and confirm `N finished units found` before Run All.
3. Save & Run All. Finished units are reused, only the rest runs. Repeat as needed, across weeks if the quota requires.

GPU T4, Internet On. **Download the log as a file before reading any number.**

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema

In [ ]:
# Find previous checkpoint dirs mounted as Inputs (resume). Empty list = fresh start.
# Kaggle mounts a notebook's Output under /kaggle/input/notebooks/<user>/<notebook>/<repo-root>/...
# (verified 2026-09-13: three levels before the repo root) — only a recursive glob finds it reliably.
import glob, os
PREV = sorted(set(glob.glob('/kaggle/input/**/ladder8-ckpt', recursive=True)))
print('resume_from:', PREV if PREV else '(none — fresh start)')
if PREV:
    n = sum(len(glob.glob(p + '/unit-*.json')) for p in PREV)
    print(f'{n} finished units found in previous versions — they will be reused, not recomputed')
else:
    print('If you expected a resume: check that the previous version is attached as Input (panel on the right).')

In [ ]:
# The run. Units are written to metrics/mqar/ladder8-ckpt/ as they finish.
# --time-budget-min 700 ≈ stop 20 min before the 12h session wall (a quota cut is handled the same way).
resume = ' '.join(f'--resume-from {p}' for p in PREV)
!python -m cortex_eval.resumable_ladder --steps 1500 --seeds 1337 2024 7 42 99 3 11 2026 --time-budget-min 700 {resume}

In [ ]:
# Status: complete (LATEST-ladder8.json) or partial (PROGRESS-ladder8.json)
import json, os
d = 'metrics/mqar/ladder8-ckpt'
if os.path.exists(d + '/LATEST-ladder8.json'):
    r = json.load(open(d + '/LATEST-ladder8.json'))
    print('COMPLETE —', r['units_done'], '/', r['units_total'], '| best pure rung:', r['best_pure_rung'])
    print('\n=== GATE READOUT (paired on seed, 95%, n>=3) ===')
    for k, v in r['gate_readout'].items(): print(f'  {k:32s} {v}')
    print('\n=== paired tests ===')
    for k, t in r['paired_tests'].items(): print(f"  {k:32s} diff {t['mean_diff']:+.4f}  t={t['t']}  df={t['df']}  crit={t['t_crit_95']}")
    print('\n=== per path: mean AUC, 95% CI ===')
    for k, st in r['stats'].items(): print(f"  {k:24s} {st['mean']:.4f}  {st['ci95']}")
    print('\nCommit metrics/mqar/ladder8-ckpt/ (all unit-*.json + LATEST) — the units ARE the record.')
else:
    r = json.load(open(d + '/PROGRESS-ladder8.json'))
    print(f"PARTIAL — {r['units_done']} / {r['units_total']} units. New version with this Output as Input → Run All to resume.")